<a href="https://colab.research.google.com/github/zFonta/CEIA-TF-Chess-DL/blob/main/notebooks/01_build_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Construcción del dataset etiquetado

Genera el dataset de pares *(posición, evaluación)* que alimenta el
entrenamiento: descarga partidas de Lichess, las filtra, muestrea posiciones y
las etiqueta con Stockfish a profundidad fija.

Cubre las tareas 3.2 a 3.6 del WBS y cierra los requerimientos 1.1, 1.2, 1.3 y 2.3.

## Cómo funciona

El pipeline hace **dos pasadas** por cada dump:

1. **Extracción** — recorre el dump comprimido (decenas de GB) una sola vez por
   streaming y guarda en un PGN chico solo las partidas que pasan el filtro. Ese
   extracto se publica en el Hub, así que esta pasada ocurre **una vez en la vida
   del proyecto**: cualquier máquina posterior lo baja en minutos en lugar de
   rehacer la hora de streaming.
2. **Etiquetado** — lee ese extracto, muestrea 4 posiciones por partida (2 con
   blancas al turno y 2 con negras), las evalúa con Stockfish y escribe shards
   Parquet, subiendo cada uno apenas se cierra.

## Dónde vive cada cosa

Todo lo durable está en Hugging Face. El disco local es solo un cache
descartable, y **no se usa Google Drive**: no hay unidad que montar, y la
corrida se puede continuar desde cualquier máquina.

| Qué | Dónde |
|---|---|
| Shards etiquetados | `ceia-chess-eval` — el entregable |
| Extracto filtrado | `ceia-chess-work` |
| Estado de reanudación | `ceia-chess-work` |
| Cache de trabajo | `/content` — se pierde y no importa |

La deduplicación no se guarda en ningún archivo: se reconstruye leyendo la
columna `pos_key` de los shards ya publicados. El dataset es su propio registro
de lo que contiene, así que no hay un segundo archivo que se desincronice.

> **Runtime: CPU, no GPU.** Stockfish es puro CPU y los runtimes con GPU de
> Colab traen *menos* vCPUs: elegir GPU acá es más lento y además gasta cuota
> que conviene reservar para el entrenamiento.

## 1. Entorno

In [2]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija (queda registrada en cada fila del dataset).
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

Listo. Directorio de trabajo: /content/CEIA-TF-Chess-DL


In [3]:
from chessdl.colab import describe_runtime

runtime = describe_runtime("stockfish")
print(runtime.summary())

Colab runtime : True
CPU workers   : 2
GPU present   : False
Stockfish     : /content/CEIA-TF-Chess-DL/bin/stockfish


## 2. Verificación del código (tests)

Antes de gastar horas de etiquetado conviene comprobar que el código hace lo que
dice. `pytest` descubre solo los archivos `tests/test_*.py`; no hay que importar
ni llamar nada a mano.

La salida de esta celda es la evidencia de los requerimientos de testing (3.1 y
3.2) para la memoria.

In [3]:
!{sys.executable} -m pytest -q

........................................................................ [ 38%]
........................................................................ [ 77%]
..........................................                               [100%]
186 passed in 29.82s


Para ver el nombre de cada test, o correr un subconjunto:

```
!{sys.executable} -m pytest tests/test_encoding.py -v     # un archivo, test por test
!{sys.executable} -m pytest -k mirror -v                  # los 3 tests del espejado
!{sys.executable} -m pytest --collect-only -q             # listar sin ejecutar
```

## 3. Credenciales de Hugging Face

El token se lee del panel de **Secrets** de Colab (icono de la llave, a la
izquierda): crear un secreto `HF_TOKEN` con permiso de **escritura** y habilitarlo
para este notebook. Nunca pegar el token en una celda.

Como ahora el Hub guarda también el extracto y el estado, sin token la corrida
no puede reanudarse en otra máquina.

In [4]:
from huggingface_hub import HfApi
from chessdl import hf
from chessdl.config import load_config

cfg = load_config()
token = hf.get_token()

if token is None:
    print("No hay token. Revisa el panel de Secrets y que el toggle de acceso")
    print("para este notebook este activado.")
else:
    info = HfApi(token=token).whoami()
    permiso = (info.get("auth", {}).get("accessToken", {}) or {}).get("role", "?")
    print("Usuario          :", info.get("name"))
    print("Permiso          :", permiso, "  <-- tiene que decir 'write'")
    print("Dataset          :", cfg.output.hf_repo_id)
    print("Datos de trabajo :", cfg.output.hf_work_repo_id)
    if info.get("name") != cfg.output.hf_namespace:
        print()
        print("OJO: el usuario del token no coincide con el namespace configurado.")

Usuario          : zFonta
Permiso          : fineGrained   <-- tiene que decir 'write'
Dataset          : zFonta/ceia-chess-eval
Datos de trabajo : zFonta/ceia-chess-work


## 4. Parámetros

Se imprimen para que queden registrados en la salida del notebook: es la
evidencia de con qué configuración se generó cada versión del dataset
(requerimiento 2.3).

In [5]:
from chessdl.data.labeling import engine_version

print("Dumps               :", cfg.source.dumps)
print("ELO minimo          :", cfg.filter.min_elo)
print("Controles de tiempo :", cfg.filter.time_controls)
print("Plies minimos       :", cfg.filter.min_plies)
print("Posiciones/partida  :", cfg.sampling.positions_per_game, "(balanceadas por turno)")
print("Semilla de muestreo :", cfg.sampling.seed)
print("Profundidad SF      :", cfg.labeling.depth)
print("Workers             :", cfg.labeling.resolved_workers())
print("Normalizacion       : value = tanh(cp /", cfg.normalization.scale,
      "), recorte +-", cfg.normalization.cp_clip)
print("Partidas por shard  :", cfg.output.games_per_shard)
print()
sf_version = engine_version(cfg.labeling)
print("Motor:", sf_version)

Dumps               : ('2025-06',)
ELO minimo          : 2200
Controles de tiempo : ('Blitz', 'Rapid', 'Classical')
Plies minimos       : 20
Posiciones/partida  : 4 (balanceadas por turno)
Semilla de muestreo : 20260909
Profundidad SF      : 12
Workers             : 2
Normalizacion       : value = tanh(cp / 400.0 ), recorte +- 2000
Partidas por shard  : 5000

Motor: Stockfish 17.1


### Cuántos workers de verdad

Colab suele reportar más CPUs de las que asigna. Si `resolved_workers()` supera
las CPUs reales, se lanzan más Stockfish de los que entran y compiten entre sí.

In [6]:
def cuota_cgroup():
    v2 = Path("/sys/fs/cgroup/cpu.max")
    if v2.exists():
        cuota, periodo = v2.read_text().split()
        if cuota != "max":
            return int(cuota) / int(periodo)
    v1q, v1p = Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us"), Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us")
    if v1q.exists() and v1p.exists():
        q = int(v1q.read_text())
        if q > 0:
            return q / int(v1p.read_text())
    return None

print("os.cpu_count()          :", os.cpu_count())
print("CPUs realmente asignadas:", len(os.sched_getaffinity(0)))
print("Cuota del cgroup        :", cuota_cgroup() or "sin limite")
print("Workers del pipeline    :", cfg.labeling.resolved_workers())

os.cpu_count()          : 2
CPUs realmente asignadas: 2
Cuota del cgroup        : sin limite
Workers del pipeline    : 2


## 5. Primera tanda

`run_build` sincroniza los shards ya publicados, reconstruye la deduplicación a
partir de ellos, trae el estado del Hub y sigue desde donde quedó.

La **primera** ejecución incluye la extracción del dump: alrededor de una hora
de streaming, sin barra de Stockfish todavía. Se publica al terminar, así que
las corridas siguientes —en esta máquina o en cualquier otra— la saltean.

`max_shards` acota cuánto se hace por sesión.

In [7]:
import time
from chessdl.data import pipeline

inicio = time.time()
resumen = pipeline.run_build(
    cfg,
    sf_version=sf_version,
    max_shards=5,
    progress=True,
)
transcurrido = time.time() - inicio

print()
print(resumen.summary())
if resumen.n_positions:
    print(f"\nVelocidad: {resumen.n_positions / transcurrido:.1f} posiciones/segundo")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

accepted from lichess_db_standard_rated_2025-06.pgn.zst: 0game [00:00, ?game/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

labelling:   0%|          | 0/17282 [00:00<?, ?pos/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...25-06_shard_00004.parquet: 100%|#########9|  830kB /  830kB            


5 shard(s) written, 88,236 positions, 10,744 duplicates skipped

Velocidad: 12.1 posiciones/segundo


### Rendimiento por shard

El techo teórico son `partidas_por_shard × posiciones_por_partida`. Lo que falta
se reparte entre partidas de menos de 20 plies y posiciones duplicadas.

In [8]:
if resumen.shards:
    print(f"{'shard':>6}{'partidas':>10}{'posiciones':>12}{'duplicadas':>12}{'descartadas':>13}")
    print("-" * 53)
    for s in resumen.shards:
        print(f"{s.index:>6}{s.n_games:>10}{s.n_positions:>12}{s.n_duplicates:>12}{s.n_dropped:>13}")

    techo = len(resumen.shards) * cfg.output.games_per_shard * cfg.sampling.positions_per_game
    print(f"\nRendimiento: {resumen.n_positions / techo:.1%} del techo teorico")
else:
    print("Ningun shard en esta tanda.")

 shard  partidas  posiciones  duplicadas  descartadas
-----------------------------------------------------
     0      5000       18224        1560           54
     1      5000       17791        2025           46
     2      5000       17519        2241           60
     3      5000       17420        2360           55
     4      5000       17282        2558           40

Rendimiento: 88.2% del techo teorico


## 6. Seguir hasta el objetivo

Este bucle repite tandas hasta llegar al objetivo o agotar el extracto. Es
**reanudable**: si Colab se desconecta, se reconecta, se corren las celdas 1 a 4
y se relanza esta misma celda.


In [7]:
from chessdl.data.state import PipelineState
from chessdl.data import pipeline

OBJETIVO = 2_500_000

while True:
    resumen = pipeline.run_build(cfg, sf_version=sf_version, max_shards=5, progress=True)
    estado = PipelineState.load(cfg.output.state_path,
                                repo_id=cfg.output.hf_work_repo_id, token=token)
    print(f"\n>>> acumulado: {estado.total_positions:,} posiciones\n")

    if not resumen.shards:
        print("Extracto agotado: no quedan partidas en este dump.")
        break
    if estado.total_positions >= OBJETIVO:
        print("Objetivo alcanzado.")
        break

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 118 files:   0%|          | 0/118 [00:00<?, ?it/s]

labelling:   0%|          | 0/15998 [00:00<?, ?pos/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

labelling:   0%|          | 0/15973 [00:00<?, ?pos/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...25-06_shard_00156.parquet:  69%|######8   |  534kB /  775kB            


>>> acumulado: 2,552,804 posiciones

Objetivo alcanzado.


## 7. Validación de integridad

Los chequeos del requerimiento 3.2 sobre lo generado hasta ahora: sin posiciones
duplicadas, balance de color, rangos válidos, sin nulos, ELO por encima del
umbral, FENs legales y no terminales, y coherencia de signo entre los dos puntos
de vista.

In [8]:
from chessdl.data import schema
from chessdl.data.validate import describe_table, validate_table

tabla = schema.read_dataset(schema.shard_paths(cfg.output.local_dir))
reporte = validate_table(tabla, cfg)
print(reporte.summary())

Dataset validation over 2,552,804 positions

[PASS] no duplicate positions: 2,552,804 distinct keys over 2,552,804 rows (0 duplicates)
[PASS] colour balance: white to move 49.68%, black to move 50.32% (limit 55%)
[PASS] label range: value_white in [-0.9999, 0.9999], no nulls or NaN
[PASS] no null values: no nulls in any column
[PASS] Elo threshold: min white 2200, min black 2200 (threshold 2200)
[PASS] sign invariant between points of view: cp and value agree on every row
[PASS] FEN legality: 5,000 FENs checked, all parseable and non-terminal
[PASS] mate flags: 36,486 mate positions, all consistent

RESULT: PASS


In [9]:
stats = describe_table(tabla)
for clave, valor in stats.items():
    print(f"{clave:>22}: {valor:,.4f}" if isinstance(valor, float) else f"{clave:>22}: {valor:,}")

           n_positions: 2,552,804
               n_games: 772,797
   white_to_move_share: 0.4968
            mate_share: 0.0143
            value_mean: 0.0463
             value_std: 0.4880
             value_min: -0.9999
             value_max: 0.9999
        mean_white_elo: 2,363.1872
        mean_black_elo: 2,363.3821
              mean_ply: 47.6571


## 8. Progreso acumulado

In [10]:
estado = PipelineState.load(cfg.output.state_path,
                            repo_id=cfg.output.hf_work_repo_id, token=token)
for nombre, dump_state in estado.dumps.items():
    print(f"{nombre}: {dump_state.shards_done} shards, "
          f"{dump_state.positions_written:,} posiciones "
          f"({dump_state.games_accepted:,} partidas aceptadas de {dump_state.games_seen:,})")
print()
print(f"Total acumulado: {estado.total_positions:,} posiciones")
print(f"Dataset        : https://huggingface.co/datasets/{cfg.output.hf_repo_id}")

2025-06: 157 shards, 2,552,804 posiciones (1,661,093 partidas aceptadas de 91,189,178)

Total acumulado: 2,552,804 posiciones
Dataset        : https://huggingface.co/datasets/zFonta/ceia-chess-eval


**Próximo paso:** `02_dataset_eda.ipynb` para el análisis exploratorio (WBS 3.5).